In [ ]:
from transformers import pipeline
import pandas as pd
import numpy as np 


data = pd.read_csv('newtwitter.csv')
data
df = data[['Content']].copy()

classifier = pipeline(
    'sentiment-analysis',
    model='cardiffnlp/twitter-roberta-base-sentiment-latest',
    tokenizer='cardiffnlp/twitter-roberta-base-sentiment-latest'
)

# Use .fillna('') to replace NaN values 
texts_to_process = df['Content'].fillna('').tolist()

num_texts = len(texts_to_process)
all_results = []
chunk_size = 200 

print(f"Starting sentiment analysis on {num_texts} texts...")

for i in range(0, num_texts, chunk_size):
    batch_texts = texts_to_process[i:i + chunk_size]

    results_batch = classifier(
        batch_texts,
        truncation=True,
        max_length=512,
        batch_size=16  
    )

    all_results.extend(results_batch)

    print(f"--- Processed {i + len(batch_texts)} / {num_texts} rows ---")


# Extract sentiment labels and add to DataFrame
df['transformer_sentiment'] = [result['label'].capitalize() for result in all_results]

print("\n--- Hugging Face Transformer Results ---")
print(df[['Content', 'transformer_sentiment']].head())

sentiment_counts = df["transformer_sentiment"].value_counts()

print("\n--- Sentiment Counts ---")
print(sentiment_counts)

sentiment_dict = sentiment_counts.to_dict()
print("\nAs dictionary:", sentiment_dict)


Some weights of the model checkpoint at cardiffnlp/twitter-roberta-base-sentiment-latest were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cpu


Starting sentiment analysis on 2001 texts...
--- Processed 200 / 2001 rows ---
--- Processed 400 / 2001 rows ---
--- Processed 600 / 2001 rows ---
--- Processed 800 / 2001 rows ---
--- Processed 1000 / 2001 rows ---
--- Processed 1200 / 2001 rows ---
--- Processed 1400 / 2001 rows ---
--- Processed 1600 / 2001 rows ---
--- Processed 1800 / 2001 rows ---
--- Processed 2000 / 2001 rows ---
--- Processed 2001 / 2001 rows ---

--- Hugging Face Transformer Results ---
                                             Content transformer_sentiment
0  What is he gonna do about it though? Jumped on...              Negative
1  In a place where every drop of water counts, I...              Negative
2                                May he rest in piss              Negative
3  i think it’s time we trend the tags again! HYB...               Neutral
4  The same question must be asked of @PrideToron...               Neutral

--- Sentiment Counts ---
transformer_sentiment
Neutral     999
Positive    644
Ne

In [ ]:
import pandas as pd
import nltk
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score

try:
    nltk.data.find('sentiment/vader_lexicon.zip')
except LookupError:
    print("Downloading VADER lexicon...")
    nltk.download('vader_lexicon')


# Load your Data 
data = pd.read_csv('newtwitter.csv')
df = pd.DataFrame(data)


# VADER Analyzer
analyzer = SentimentIntensityAnalyzer()

def get_vader_sentiment(text):
    """
    Classifies text as Positive, Negative, or Neutral
    based on VADER's 'compound' score.
    """
    scores = analyzer.polarity_scores(text)
    compound_score = scores['compound']
    
    if compound_score >= 0.05:
        return 'Positive'
    elif compound_score <= -0.05:
        return 'Negative'
    else:
        return 'Neutral'

#Apply VADER to your DataFrame 
df['Content_Clean'] = df['Content'].fillna('')
df['vader_sentiment'] = df['Content_Clean'].apply(get_vader_sentiment)


print("--- VADER Sentiment Analysis Results ---")
print(df[['Content', 'vader_sentiment']].head(10))

print("\n--- VADER Sentiment Counts ---")
print(df['vader_sentiment'].value_counts())
print("-" * 40)


X = df['Content_Clean']
y = df['vader_sentiment']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y 
)

print(f"Total samples: {len(df)}")
print(f"Training samples: {len(X_train)}")
print(f"Testing samples: {len(X_test)}")
print("-" * 40)


# Train and Evaluate LinearSVC
print("Training LinearSVC model to mimic VADER...")

# Create a TF-IDF Vectorizer
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))

X_train_vec = vectorizer.fit_transform(X_train)

X_test_vec = vectorizer.transform(X_test)

svc_model = LinearSVC(dual="auto") 
svc_model.fit(X_train_vec, y_train)

y_pred_svc = svc_model.predict(X_test_vec)


acc_svc = accuracy_score(y_test, y_pred_svc)

print("\n--- Model Evaluation ---")
print(f"Target: VADER's predictions")
print(f"Model: LinearSVC")
print(f"Accuracy: {acc_svc * 100:.2f}%")
print("\nThis accuracy score represents how well the LinearSVC model")
print("was able to learn and replicate VADER's sentiment labels.")

--- VADER Sentiment Analysis Results ---
                                             Content vader_sentiment
0  What is he gonna do about it though? Jumped on...        Negative
1  In a place where every drop of water counts, I...        Negative
2                                May he rest in piss        Negative
3  i think it’s time we trend the tags again! HYB...        Positive
4  The same question must be asked of @PrideToron...        Positive
5  I organized this event at UCSD in May 2010. No...        Negative
6  He didnt reflect on his actions What tweets wo...        Positive
7  The student uprisings for Palestine have gone ...        Negative
8  Happening now: Thousands march the streets of ...        Negative
9  “When it's nobody's business the way that you ...        Positive

--- VADER Sentiment Counts ---
vader_sentiment
Positive    850
Neutral     652
Negative    499
Name: count, dtype: int64
----------------------------------------
Total samples: 2001
Training samples: